In [1]:
import torch
import torch.nn as nn
from torch import tensor
import torch.nn.functional as F

### Not using kv cache

In [23]:
attention = nn.MultiheadAttention(
    embed_dim=2,
    num_heads=1,
    batch_first=True,
    bias=False
)

attention.load_state_dict({
    "in_proj_weight": torch.tensor([
        # W_Q
        [0.1, 0.2],
        [0.3, 0.4],

        # W_K
        [0.2, 0.3],
        [0.4, 0.5],

        # W_V
        [0.3, 0.4],
        [0.5, 0.6],
    ]),
    
    "out_proj.weight": torch.tensor([
        [1.0, 1.1],
        [1.2, 1.3],
    ])
})

x = torch.tensor([
    [
        [0.2, 0.3],
        [0.4, 0.5],
        [0.5, 0.6],
    ]
])

In [24]:
output, weights = attention(
    query=x,
    key=x,
    value=x,
)

In [25]:
output[0][-1]

tensor([0.8154, 0.9691], grad_fn=<SelectBackward0>)

In [26]:
x = torch.tensor([
    [
        [0.2, 0.3],
        [0.4, 0.5],
        [0.5, 0.6],
        [0.6, 0.7], # New Token
    ]
])

output, weights = attention(
    query=x,
    key=x,
    value=x,
)
output[0][-1]

tensor([0.9327, 1.1086], grad_fn=<SelectBackward0>)

### Using kv cache

In [27]:
attention = nn.MultiheadAttention(
    embed_dim=2,
    num_heads=1,
    batch_first=True,
    bias=False
)

attention.load_state_dict({
    "in_proj_weight": torch.tensor([
        # W_Q
        [0.1, 0.2],
        [0.3, 0.4],

        # W_K
        [0.2, 0.3],
        [0.4, 0.5],

        # W_V
        [0.3, 0.4],
        [0.5, 0.6],
    ]),

    "out_proj.weight": torch.tensor([
        [1.0, 1.1],
        [1.2, 1.3],
    ])
})

<All keys matched successfully>

In [36]:
W_Q = attention.in_proj_weight[:2]
W_K = attention.in_proj_weight[2:4]
W_V = attention.in_proj_weight[4:6]

W_O = attention.out_proj.weight

In [44]:
## Calculate x_old

x = torch.tensor([
    [
        [0.2, 0.3],
        [0.4, 0.5],
        [0.5, 0.6],
    ]
])

Q = x @ W_Q.T
K = x @ W_K.T
V = x @ W_V.T
score = Q @ K.transpose(-2, -1)
score_scaled = score / torch.sqrt(torch.tensor(2.0))
weights = F.softmax(score_scaled, dim=-1)
attn = weights @ V
output = attn @ W_O.T
output[0][-1]

tensor([0.8154, 0.9691], grad_fn=<SelectBackward0>)

In [45]:
## Calculate new token

x_new = torch.tensor([
    [
        [0.6, 0.7]
    ]
])

K_cache = K.clone()
V_cache = V.clone()

Q_new = x_new @ W_Q.T
K_new = x_new @ W_K.T
V_new = x_new @ W_V.T

K_cache = torch.cat([K_cache, K_new], dim=1)
V_cache = torch.cat([V_cache, V_new], dim=1)

score = Q_new @ K_cache.transpose(-2, -1)
score_scaled = score / torch.sqrt(torch.tensor(2.0))
weights = F.softmax(scores, dim=-1)
attn = weights @ V_cache
output = attn @ W_O.T
output[0][-1]

tensor([0.9327, 1.1086], grad_fn=<SelectBackward0>)